# Türkçe Duygu Analizi — DistilBERT
Google Colab ile üç sınıflı duygu analizi. Hücreleri sırayla çalıştırın. Önceden raporlanan sonuçlar `results/experiment_summary.md` dosyasındadır; yeniden çalıştırmada değişebilir.

## 1. Paketler

In [ ]:
%pip install -q "transformers==4.46.3" "accelerate>=0.26.0" datasets scikit-learn pandas matplotlib

## 2. Veri hazırlama

In [ ]:
import torch, pandas as pd, numpy as np
from datasets import load_dataset, Dataset
from sklearn.model_selection import train_test_split
dataset = load_dataset("winvoker/turkish-sentiment-analysis-dataset")
df = dataset["train"].to_pandas()
balanced_df = df.groupby("label", group_keys=False).sample(n=2000, random_state=42).reset_index(drop=True)
label2id = {"Negative": 0, "Notr": 1, "Positive": 2}
id2label = {v: k for k, v in label2id.items()}
balanced_df["label"] = balanced_df["label"].map(label2id)
train_df, val_df = train_test_split(balanced_df, test_size=0.2, stratify=balanced_df["label"], random_state=42)
train_data = Dataset.from_pandas(train_df[["text","label"]].reset_index(drop=True), preserve_index=False)
val_data = Dataset.from_pandas(val_df[["text","label"]].reset_index(drop=True), preserve_index=False)
print("Training:",len(train_data),"Validation:",len(val_data),"GPU:",torch.cuda.is_available())

## 3. Model ve tokenization

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
model_name = "distilbert/distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=3, id2label=id2label, label2id=label2id)
def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True, max_length=128)
train_tokenized = train_data.map(tokenize_function, batched=True)
val_tokenized = val_data.map(tokenize_function, batched=True)

## 4. Fine-tuning

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    pred = np.argmax(logits, axis=1)
    return {"accuracy": accuracy_score(labels,pred), "f1_macro": f1_score(labels,pred,average="macro")}
args = TrainingArguments(
    output_dir="./turkish_sentiment_model", num_train_epochs=3,
    learning_rate=2e-5, per_device_train_batch_size=8, per_device_eval_batch_size=16,
    warmup_steps=60, weight_decay=0.01, eval_strategy="epoch", save_strategy="epoch",
    logging_strategy="epoch", load_best_model_at_end=True,
    metric_for_best_model="f1_macro", greater_is_better=True,
    fp16=torch.cuda.is_available(), report_to="none", save_total_limit=1, seed=42)
trainer = Trainer(model=model,args=args,train_dataset=train_tokenized,eval_dataset=val_tokenized,
    tokenizer=tokenizer,data_collator=DataCollatorWithPadding(tokenizer=tokenizer),compute_metrics=compute_metrics)
trainer.train()

## 5. Eğitim eğrileri ve model kaydı

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
Path("results").mkdir(exist_ok=True)
trainer.save_model("final_model")
tokenizer.save_pretrained("final_model")
history = pd.DataFrame(trainer.state.log_history)
history.to_csv("results/training_history.csv", index=False)
train_hist = history.dropna(subset=["loss"])
eval_hist = history.dropna(subset=["eval_loss"])
plt.figure(figsize=(8,5))
plt.plot(train_hist["epoch"],train_hist["loss"],marker="o",label="Training Loss")
plt.plot(eval_hist["epoch"],eval_hist["eval_loss"],marker="s",label="Validation Loss")
plt.xlabel("Epoch");plt.ylabel("Loss");plt.title("DistilBERT Fine-Tuning Loss")
plt.legend();plt.grid(True)
plt.savefig("results/training_loss.png",dpi=300,bbox_inches="tight");plt.show()

## 6. Bağımsız test değerlendirmesi

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
test_df = dataset["test"].to_pandas()
test_df, _ = train_test_split(test_df, train_size=3000, stratify=test_df["label"], random_state=42)
test_df["label"] = test_df["label"].map(label2id)
test_data = Dataset.from_pandas(test_df[["text","label"]].reset_index(drop=True), preserve_index=False)
test_tokenized = test_data.map(tokenize_function,batched=True)
out = trainer.predict(test_tokenized)
y_true, y_pred = out.label_ids, out.predictions.argmax(axis=1)
report = classification_report(y_true,y_pred,labels=[0,1,2],target_names=["Negative","Notr","Positive"],digits=4)
print(report)
Path("results/classification_report.txt").write_text(report,encoding="utf-8")
cm = confusion_matrix(y_true,y_pred,labels=[0,1,2])
disp = ConfusionMatrixDisplay(confusion_matrix=cm,display_labels=["Negative","Notr","Positive"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix - Turkish Sentiment Analysis")
plt.savefig("results/confusion_matrix.png",dpi=300,bbox_inches="tight");plt.show()

## 7. Yeni cümleler

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(text,return_tensors="pt",truncation=True,max_length=128).to(trainer.model.device)
    trainer.model.eval()
    with torch.no_grad():
        logits = trainer.model(**inputs).logits
    probs = torch.softmax(logits,dim=1)[0]
    idx = int(probs.argmax().item())
    return id2label[idx],float(probs[idx].item())
for text in ["Bu ürünü çok beğendim, harika çalışıyor.","Ürün çok kötü, hiç memnun kalmadım.","Paket bugün teslim edildi."]:
    label,confidence = predict_sentiment(text)
    print(f"{text} → {label} ({confidence:.2%})")